In [ ]:
import librosa.display
import torch
import torch.nn as nn

import IPython.display as ipd  # used to display audio
import json
import yaml  # used to read config files of RawNet2
import time  # used to measure training time
import shutil  # used to save checkpoint
import random
import AASIST
from Model import  DownStreamLinearClassifier, RawNetEncoderBaseline, MoCo_v2
from DatasetUtils import genSpoof_list, Dataset_ASVspoof2019_train, MoCoAudioDataset,  VolumeChange, AddWhiteNoise, AddEnvironmentalNoise, WaveTimeStretch, AddEchoes, TimeShift, AddFade, ResampleAugmentation  # ASVspoof dataset utils, MoCo dataset utils, and data augmentation utils
import torchvision.transforms as transforms

# set random seed
torch.manual_seed(0)
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
# parameters
gpu = 1  # gpu id to use
margin = 4  # margin used by length loss
alpha = 2  # alpha used by length loss
real_weight = 9.0  # weight used in length loss because of the unbalanced dataset
weight_decay = 1e-4  # weight decay for optimizer
batch_size = 24
moco_k = 6144  # queue size of MoCo framework
moco_t = 0.07  # temperature of MoCo framework
moco_m = 0.999  # Momentum of MoCo framework
print_freq = 10  # print frequency during training
pretrain_lr = 0.0005  # learning rate
pretrain_epochs = 150  # Pretraining epochs number
downstream_lr = 0.0001  # downstream learning rate
downstream_epochs = 10  # downstream training epochs number

feature_dim = 160  # feature dimension number output by the encoder
cos_lr_decay = True  # cosine annealing learning rate decay
use_length_loss = True  # if use length loss function
mlp = False  # if use projection head in Moco framework
encoder_name = "AASIST"  # encoder name

torch.cuda.set_device(gpu)
# Load Config file
with open("./config.conf", "r") as f_json:
    config = json.loads(f_json.read())

In [ ]:
# Initialize the MoCo model to train
'''
encoder_name: name of the encoder architecture to use, can be Rawnet2 or AASIST
config: configuration read from the file
''' 
def init_moco_model(encoder_name:str, config: dict):
    if encoder_name == "Rawnet2":
        dir_yaml = config['rawnet2_config_path']
        with open(dir_yaml, 'r') as f_yaml:
                parser1 = yaml.safe_load(f_yaml)
        encoder_q = RawNetEncoderBaseline(parser1['model'], device)
        encoder_k = RawNetEncoderBaseline(parser1['model'], device)
        encoder_q =(encoder_q).to(device)
        encoder_k =(encoder_k).to(device)
        nb_params = sum([param.view(-1).size()[0] for param in encoder_q.parameters()])
        print(f"Number of Rawnet2 encoder params:{nb_params}")
        queue_feature_dim = 1024
    if encoder_name == "AASIST":
        with open(config['aasist_config_path'], "r") as f_json:
            config = json.loads(f_json.read())
        model_config = config["model_config"]
        encoder_q = AASIST.AasistEncoder(model_config).to(device)
        encoder_k = AASIST.AasistEncoder(model_config).to(device)
        nb_params = sum([param.view(-1).size()[0] for param in encoder_q.parameters()])
        print("Number of AASIST encoder params:{}".format(nb_params))
        queue_feature_dim = 160
    moco_v2 = MoCo_v2(encoder_q=encoder_q, encoder_k = encoder_k, queue_feature_dim=queue_feature_dim,queue_size=moco_k, momentum=moco_m, temperature=moco_t, mlp=mlp, return_q=use_length_loss).to(device)
    return moco_v2
        



In [ ]:
moco_v2 = init_moco_model("AASIST", config)

In [ ]:
# define loss function (criterion) and optimizer
criterion = nn.CrossEntropyLoss().cuda(gpu)

if mlp:
    params_to_optimize = [{'params': moco_v2.encoder_q.parameters()}, {'params': moco_v2.projection_head_q.parameters()}]
else:
    params_to_optimize = moco_v2.encoder_q.parameters()

optimizer = torch.optim.Adam(
    params=params_to_optimize,
    lr = pretrain_lr,
    weight_decay=weight_decay,
)

In [ ]:
# Initialize moco_asvspoof19LA_training_dataset
database_path=config['database_path']
d_label_trn, file_train = genSpoof_list(dir_meta=database_path+"ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt", is_train=True, is_eval=False)
# file_train = [x + '.flac' for x in file_train]  # add ".flac" suffix to every string item in file_train
print('no. of training trials', len(file_train))
moco_asvspoof19LA_training_dataset = MoCoAudioDataset(root_dir=database_path+"ASVspoof2019_LA_train/flac/", file_list=file_train, label_list=d_label_trn, transform=None)
# Initialize asvspoof19LA_training_dataloader for pretraining
moco_asvspoof19LA_training_dataloader = torch.utils.data.DataLoader(moco_asvspoof19LA_training_dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, drop_last=True)
# Initialize asvspoof19LA_training_dataloader for downstream training
downstream_asvspoof19LA_training_dataset = Dataset_ASVspoof2019_train(list_IDs=file_train, labels=d_label_trn, base_dir=database_path+"ASVspoof2019_LA_train/")
downstream_asvspoof19LA_training_dataloader = torch.utils.data.DataLoader(downstream_asvspoof19LA_training_dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, drop_last=True)


In [ ]:
def weighted_length_loss(feature_vectors, y, margin, real_weight=1):
    # for y=1, the bonafide sample, we expect the length to be zero, while for y=0, the fake sample,we expect the length is larger than margin
    lengths = torch.norm(feature_vectors, p=2, dim=1) # calculate the length of each feature vector using L2-norm
    # print(lengths)
    # loss = (1-y) * torch.clamp(margin-lengths,min=0)
    # loss = y*lengths
    loss = y*lengths*real_weight + (1-y) * torch.clamp(margin-lengths,min=0)
    # print(loss)
    loss = torch.mean(loss) # calculate the mean of the absolute differences as the loss
    return loss

'''
Train the MoCo model, the training process is similar to the training process of MoCo
'''
def train_moco(train_loader, model, criterion, optimizer, epoch, alpha, margin, real_weight, print_freq, start_epoch, augmentations=None):
    batch_time = AverageMeter("Time", ":6.3f")
    data_time = AverageMeter("Data", ":6.3f")
    model_time = AverageMeter("Model", ":6.3f")
    losses = AverageMeter("Loss", ":.4e")
    top1 = AverageMeter("Acc@1", ":6.2f")
    top5 = AverageMeter("Acc@5", ":6.2f")
    progress = ProgressMeter(
        len(train_loader),
        [batch_time, data_time, model_time, losses, top1, top5],
        prefix="Epoch: [{}]".format(epoch),
    )

    # switch to train mode
    model.train()

    end = time.time()
    for i, (audio1, audio2, label) in enumerate(train_loader):
        

        if torch.cuda.is_available():
            # put audio on cuda
            audio1 = audio1.cuda(gpu, non_blocking=True)
            audio2 = audio2.cuda(gpu, non_blocking=True)
            label = label.cuda(gpu, non_blocking=True)
        if augmentations != None:
            audio1 = augmentations(audio1)
            audio2 = augmentations(audio2)
            # audio1 = pad_or_clip(audio1, 64600)
            # audio2 = pad_or_clip(audio2, 64600)
        # measure data loading time
        data_time_tmp = time.time() - end
        data_time.update(data_time_tmp)
        
        # compute output
        output, target, q_embeddings = model(x_q=audio1, x_k=audio2)
        model_time_tmp = time.time() - end - data_time_tmp
        model_time.update(model_time_tmp)


        loss = criterion(output, target) + alpha * weighted_length_loss(q_embeddings, label, margin, real_weight)
        # loss = criterion(output, target)

        # acc1/acc5 are (K+1)-way *contrastclassifier accuracy*
        # measure accuracy and record loss
        acc1, acc5 = accuracy(output, target, topk=(1, 5))
        losses.update(loss.item(), audio1.size(0))
        top1.update(acc1[0], audio1.size(0))
        top5.update(acc5[0], audio1.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()

        if i % print_freq == 0 or (i < 10 and epoch == start_epoch):  # modified here, to print early progress
            progress.display(i)


def save_checkpoint(state, is_best, filename="checkpoint.pth.tar"):
    torch.save(state, filename)
    if is_best:
        shutil.copyfile(filename, "model_best.pth.tar")


class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self, name, fmt=":f"):
        self.name = name
        self.fmt = fmt
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def __str__(self):
        fmtstr = "{name} {val" + self.fmt + "} ({avg" + self.fmt + "})"
        return fmtstr.format(**self.__dict__)


class ProgressMeter(object):
    def __init__(self, num_batches, meters, prefix=""):
        self.batch_fmtstr = self._get_batch_fmtstr(num_batches)
        self.meters = meters
        self.prefix = prefix

    def display(self, batch):
        entries = [self.prefix + self.batch_fmtstr.format(batch)]
        entries += [str(meter) for meter in self.meters]
        print("\t".join(entries))

    def _get_batch_fmtstr(self, num_batches):
        num_digits = len(str(num_batches // 1))
        fmt = "{:" + str(num_digits) + "d}"
        return "[" + fmt + "/" + fmt.format(num_batches) + "]"

# below are some utils

import math
def adjust_learning_rate(optimizer, epoch, original_lr, epochs_to_train, cos_lr_decay, schedule=None, min_lr=5e-6):
    """Decay the learning rate based on schedule"""
    lr = original_lr
    if cos_lr_decay:  # cosine lr schedule
        lr *= 0.5 * (1.0 + math.cos(math.pi * epoch / epochs_to_train))
    else:  # stepwise lr schedule
        if schedule == None:
            schedule = [epochs_to_train * 0.5, epochs_to_train * 0.75]
        for milestone in schedule:
            lr *= 0.1 if epoch >= milestone else 1.0
    lr = max(lr, min_lr)  # added in exp20, to make sure the learning rate is not too low.
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

def accuracy(output, target, topk=(1,)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

def pad_or_clip(audio, audio_len):
        '''
        Pad or randomly clip the audio to make it of length audio_len
        '''
        if audio.shape[-1] < audio_len:
            audio = torch.nn.functional.pad(audio, (0, audio_len - audio.shape[-1]))
        elif audio.shape[-1] > audio_len:
            # randomly clip the audio
            start = random.randint(0, audio.shape[-1] - audio_len)
            # start = 0 # uncomment this line to remove random clipping
            audio = audio[start:start+ audio_len]
        return audio

In [ ]:
moco_augmentations = transforms.Compose([
    transforms.RandomApply([
        transforms.RandomChoice([
            AddWhiteNoise(max_snr_db=25, min_snr_db=10),
            AddEnvironmentalNoise(max_snr_db=25, min_snr_db=10, noise_dataset_path = config['noise_dataset_path'],device="cuda")
        ])
    ], p=0.3),
    transforms.RandomApply([
        WaveTimeStretch(max_ratio=1.15,min_ratio=0.85, n_fft=128)  # exp 20 nfft=128
    ], p=0.3),
    transforms.RandomApply([
        VolumeChange(max_vol=0.2, min_vol=0.01)
    ], p=0.3),
    transforms.RandomApply([
        TimeShift(max_shift=64000, min_shift=0)
    ], p=0.3),
    transforms.RandomApply([
        # AddFade(max_fade_size=.5, fade_shape="exponential")
        AddFade(max_fade_size=.5, fix_fade_size=True)
    ], p=0.3),
    transforms.RandomApply([
        ResampleAugmentation([14500, 17500])
    ], p=0.1),
    # transforms.RandomApply([
    #     SmoothingAugmentation()
    # ], p=0.1),
    transforms.RandomApply([
        AddEchoes(max_delay=3200, min_delay=1, max_strengh=0.5, min_strength=0.3)  # when the delay is 0, audio[:-0] will get a void tensor.
    ], p=0.3),
    
])

In [ ]:
# training loop
for epoch in range(0, pretrain_epochs):
# for epoch in range(3):
    adjust_learning_rate(optimizer, epoch, pretrain_lr, pretrain_epochs, cos_lr_decay)
    # train for one epoch
    train_moco(moco_asvspoof19LA_training_dataloader, moco_v2, criterion, optimizer, epoch, alpha, margin, real_weight, print_freq, 0, augmentations=moco_augmentations)
    # save checkpoint
    if epoch % 20 == 0:
        save_checkpoint(
            {
                "epoch": epoch+1,
                "state_dict": moco_v2.state_dict(),
                "moco_queue": moco_v2.queue,
            },
            is_best=False,
            filename="MoCo_{}_epoch{:04d}.pth.tar".format(encoder_name, epoch+1),
        )
# save the final model
save_checkpoint(
    {
        "epoch": epoch+1,
        "state_dict": moco_v2.state_dict(),
        "moco_queue": moco_v2.queue,
    },
    is_best=False,
    filename="MoCo_{}_epoch{:04d}.pth.tar".format(encoder_name, epoch+1),
)

In [ ]:
# Downstream training
# Initialize the downstream model
'''
encoder_name: name of the encoder architecture to use, can be Rawnet2 or AASIST
config: configuration read from the file
moco_v2: the pretrained MoCo model
'''
def init_downstream_model(encoder_name:str, config: dict, feature_dim:int, moco_v2: MoCo_v2):
    if encoder_name == "Rawnet2":
        dir_yaml = config['rawnet2_config_path']
        with open(dir_yaml, 'r') as f_yaml:
                parser1 = yaml.safe_load(f_yaml)
        encoder = RawNetEncoderBaseline(parser1['model'], device)
        encoder =(encoder).to(device)
        nb_params = sum([param.view(-1).size()[0] for param in encoder.parameters()])
        print(f"Number of Rawnet2 encoder params:{nb_params}")
    if encoder_name == "AASIST":
        with open(config['aasist_config_path'], "r") as f_json:
            config = json.loads(f_json.read())
        model_config = config["model_config"]
        encoder = AASIST.AasistEncoder(model_config).to(device)
        nb_params = sum([param.view(-1).size()[0] for param in encoder.parameters()])
        print("Number of AASIST encoder params:{}".format(nb_params))
    encoder.load_state_dict(moco_v2.encoder_q.state_dict())    # load the pretrained encoder weights
    downstream_model = DownStreamLinearClassifier(encoder=encoder, input_depth=feature_dim).to(device)
    return downstream_model
downstream_model = init_downstream_model(encoder_name, config, feature_dim, moco_v2)

In [ ]:
for param in downstream_model.encoder.parameters():
    param.requires_grad = False

In [ ]:
def train_downstream(train_loader, model, downstream_criterion, optimizer, epoch, gpu, print_freq):
    batch_time = AverageMeter("Time", ":6.3f")
    data_time = AverageMeter("Data", ":6.3f")
    losses = AverageMeter("Loss", ":.4e")
    # top1 = AverageMeter("Acc@1", ":6.2f")
    progress = ProgressMeter(
        len(train_loader),
        [batch_time, data_time, losses],
        prefix="Epoch: [{}]".format(epoch),
    )

    # switch to train mode
    model.train()

    end = time.time()
    for i, (audio, target) in enumerate(train_loader):
        # measure data loading time
        data_time.update(time.time() - end)
        # target = F.one_hot(target, num_classes=2)
        # the audio has shape (batch_size, 1, audio_len), we need to squeeze it.
        audio = audio.squeeze(1)
        if torch.cuda.is_available():
            # put data on cuda
            audio = audio.cuda(gpu, non_blocking=True)
            target = target.cuda(gpu, non_blocking=True)

        # compute output
        output = model(audio)
        loss = downstream_criterion(output, target)

        # print(target)

 
        losses.update(loss.item(), audio.size(0))
        # top1.update(acc1[0], audio.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()

        if i % print_freq == 0 or (i < 10 and epoch == 0):  # modified here, to print early progress
            progress.display(i)



In [ ]:
# define loss function (criterion) and optimizer
weight = torch.FloatTensor([0.1, 0.9]).to(device)  # Note that we set weight here for downstream criterion!
downstream_criterion = nn.CrossEntropyLoss(weight=weight)
# downstream_criterion(torch.tensor([1.0, 2.0, 3.0]).cuda(), torch.tensor([1.0, 2.0, 3.0]).cuda())
optimizer = torch.optim.Adam(
    # params=[{'params': downstream_model.fc1.parameters()}, {'params': downstream_model.fc2.parameters()}],
    params=[{'params': downstream_model.fc.parameters()}],  # used for single layer classifier
    lr=downstream_lr,
    weight_decay=weight_decay
)


In [ ]:
# training loop
for epoch in range(0, downstream_epochs):
    # adjust_learning_rate(optimizer, epoch, args)
    is_best = False
    train_downstream(downstream_asvspoof19LA_training_dataloader, downstream_model, downstream_criterion, optimizer, epoch, gpu, print_freq)
    save_checkpoint(
        {
            "epoch": epoch,
            "state_dict": downstream_model.state_dict(),
            "optimizer": optimizer.state_dict(),
        },
        is_best=is_best,
        filename="{}_pretrained{}__downstream_checkpoint_{:04d}.pth.tar".format(encoder_name, pretrain_epochs, epoch),
    )


